In [1]:
import sys
print(sys.executable)
print(sys.version)

/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/.venv/bin/python3.9
3.9.6 (default, May 22 2026, 11:13:45) 
[Clang 21.0.0 (clang-2100.1.1.101)]


# Trassenfinder API Query

This notebook explores the Deutsche Bahn Trassenfinder API as a data source for the night train energy model.

The goal is to collect real route data including:
- distance
- travel time
- train parameters
- energy consumption

The collected data will later be used to calibrate a regression-based energy model.

## Workflow

1. Connect to the Trassenfinder API
2. Test a single route query
3. Inspect the API response structure
4. Extract relevant energy model variables
5. Automate multiple route queries
6. Save collected data as a training dataset

## API Information

Source:
Deutsche Bahn Trassenfinder API

Purpose:
Retrieve railway route information and energy consumption estimates.

Authentication:
No API key required.

The API uses station identifiers (DS100 codes) to define origin and destination.

In [4]:
# ============================================================
# SETUP – Trassenfinder Energy Model
# ============================================================

import pandas as pd
import requests

# ------------------------------------------------------------
# 1. CSV-Dateien laden
# Die Dateien liegen in: energy/data/raw/
# Das Notebook liegt in: energy/notebooks/
# ------------------------------------------------------------

compositions = pd.read_csv(
    "../data/raw/compositions_for_energy_model.csv"
)

routes = pd.read_csv(
    "../data/raw/ontd_segments_germany.csv"
)

# ------------------------------------------------------------
# 2. Erste Komposition für den Test auswählen
# ------------------------------------------------------------

composition = compositions.iloc[0]

# ------------------------------------------------------------
# 3. Trassenfinder API
# ------------------------------------------------------------

url = "https://trassenfinder.de/api/web/routen/suche"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Origin": "https://trassenfinder.de",
    "Referer": "https://trassenfinder.de/",
}

# ------------------------------------------------------------
# 4. Kontrolle
# ------------------------------------------------------------

print("✓ Setup erfolgreich")
print()
print("Komposition:", composition["composition_id"])
print("Gewicht:", composition["coaches_gross_weight_80pct_t_wagenzugmasse"], "t")
print("Länge:", composition["coaches_length_m_wagenzuglaenge"], "m")
print("Vmax:", composition["v_max_kmh"], "km/h")
print("Lok:", composition["trassenfinder_triebfahrzeug_hauptnummer"])
print()
print("Anzahl Kompositionen:", len(compositions))
print("Anzahl Nachtzug-Abschnitte:", len(routes))

✓ Setup erfolgreich

Komposition: REF-BUD-6
Gewicht: 323.3 t
Länge: 158.4 m
Vmax: 200 km/h
Lok: 6193

Anzahl Kompositionen: 8
Anzahl Nachtzug-Abschnitte: 96


In [7]:
# ============================================================
# TEST 1 – Eine Nachtzug-Route mit einer Komposition
# ============================================================

# Erste Route aus der Nachtzug-CSV
route = routes.iloc[0]

# ------------------------------------------------------------
# Payload für Schienenpersonenfernverkehr mit Lok
# ------------------------------------------------------------

# ============================================================
# TEST 1 – SPV-Payload ohne Trassenentgelt-Marktsegment
# ============================================================

route = routes.iloc[0]
composition = compositions.iloc[0]

payload = {
    "infrastruktur_id": 8,

    "sucheinstellungen": {
        "verkehrsart": "spfv_lok",
        "an_abzeit": "2026-08-12T20:00:00+02:00",
        "zeitvorgabe_typ": "abzeit",

        "optimierungsvarianten_berechnen": True,
        "richtungswechsel_zulaessig": True,
        "rangierfahrt_zulaessig": False,

        "vermeidung_parameter": {
            "ueberlastete_meiden": False,
            "sbahnen_meiden": True,
            "nebenbahnen_meiden": False,
            "schnellfahrstrecken_meiden": True,
            "knotenbahnhoefe_meiden": True,
            "notbremsueberbrueckung_meiden": False,
            "wirbelstrombremse_meiden": False,
            "eingleisige_strecken_meiden": False,
            "strecken_mit_vorrang_sgv_meiden": False,
            "strecken_mit_vorrang_spv_meiden": False
        },

        "initiale_sperrungen_beruecksichtigen": True,
        "wendezeit_min": 30,
        "mit_realistischen_fahrzeiten_optimieren": True,
        "verkehrshalte_nur_an_bahnsteigen": False,
        "einschraenkungen_beachten": True,

        "manueller_fahrzeitzuschlag_prozent": 0,
        "einzelgrenzlastberechnung_zulaessig": False,
        "bauzuschlaege_beachten": False,
        "laengenabhaengige_grenzlasten_verwenden": False,
        "tpn_triebfahrzeugbezeichnung_anzeigen": False,

        "gewichtung_parameter": {
            "streckenlaenge_prozent": 40,
            "fahrzeit_prozent": 30,
            "energie_prozent": 30
        },

        "zusatzkosten_parameter": {
            "energiebezugspreis_euro_pro_kwh": 0.18,
            "rueckspeisung_euro_pro_kwh": 0.09,
            "kosten_besetzte_tfz_inkl_personal_euro_pro_h": 150,
            "kosten_unbesetzte_tfz_euro_pro_h": 80,
            "kosten_wagenzug_euro_pro_h": 150,
            "kostenpauschale_ungekuppelt_nachschieben_euro": 999,
            "zusaetzlicher_energieverbrauch_pro_wagen_kw": 0,
            "energieverbrauch_hilfsbetriebe_und_wagen_beachten": True
        }
    },

    "wegpunkte": [
        {
            "zugcharakteristik": {
                "bremshundertstel": 70,
                "bremsstellung": "P",
                "aktive_neigetechnik": False,
                "kupplungsbauart": "kn450",
                "dla_u_profile": [],
                "fuehrendes_fahrzeug": "lokomotive",

                "kv_profil": {
                    "p": "N",
                    "c": "N"
                },

                "nachschiebeart": "ohne",
                "streckenklasse": "D4",
                "traktionsartwechsel": False,

                "triebfahrzeug": {
                    "hauptnummer": str(
                        composition["trassenfinder_triebfahrzeug_hauptnummer"]
                    ),
                    "unternummer": 2,
                    "kennung": "L",
                    "kennung_wert": 80
                },

                "vorspannart": "ohne",

                "wagenzuglaenge_m": float(
                    composition["coaches_length_m_wagenzuglaenge"]
                ),

                "wagenzugmasse_t": float(
                    composition["coaches_gross_weight_80pct_t_wagenzugmasse"]
                ),

                "wagenanzahl": int(
                    composition["n_coaches"]
                ),

                "v_max": float(
                    composition["v_max_kmh"]
                ),

                "zugbeeinflussung_parameter": {
                    "etcs_system_version": "ohne",
                    "lzb": True,
                    "pzb": True
                }
            },

            "betriebsstelle": {
                "ds100": str(route["start_ds100"]),
                "mutter": True
            }
        },

        {
            "betriebsstelle": {
                "ds100": str(route["end_ds100"]),
                "mutter": True
            }
        }
    ],

    "nutzer_sperrungen": []
}

print("✓ Payload erstellt")
print("Route:", route["start_stop_name"], "→", route["end_stop_name"])
print("Komposition:", composition["composition_id"])

✓ Payload erstellt
Route: Augsburg Hbf → Hannover Hbf
Komposition: REF-BUD-6


In [6]:
# ============================================================
# TEST 1 – Anfrage an Trassenfinder
# ============================================================

response = requests.post(
    url,
    json=payload,
    headers=headers
)

print("Status:", response.status_code)
print("Antwort:")
print(response.text[:2000])

Status: 400
Antwort:
{"message":"Ungültiger Wert für Element 'sucheinstellungen.trassenentgelt_parameter.marktsegment'","details":[],"error_type":"invalid_request"}


In [8]:
response = requests.post(
    url,
    json=payload,
    headers=headers
)

print("Status:", response.status_code)
print(response.text[:2000])

Status: 200
{"result":{"id":"d905097f6220dae4","gewichtete_route":{"routen_typ":"gewichtete","routenpunkte":[{"ds100":"MAOB","wegpunkt_index":0,"koordinate":{"x":41773.7723538518,"y":55887.209339679684},"naechstes_streckensegment":{"von":"MAOB","bis":"MGHF","streckennummer":5300},"laufende_hm":0,"technische_fahrzeit_info":{"ankunft_min":0,"abfahrt_min":0},"geschwindigkeit_technisch_hmh":943,"haltart":"kundenhalt","halteplatz_sprungart":"durchbindung","schiebelok_kupplungsart":"nichts","verkehrshalt_trotz_fehlendem_bahnsteig":false,"halteplatz_zu_kurz":false,"gleiswechselbetrieb":false,"begegnungsverbot":false,"energieverbrauch_info":{"energieverbrauch_traktion_kwh":0,"energieverbrauch_hilfsbetriebe_kwh":0,"energieverbrauch_wagen_kwh":0,"energieverbrauch_gesamt_kwh":0},"energieverbrauch_kosten_euro":0,"marktsegmente":[],"trassenpreis_euro":0,"stationspreis_euro":8,"kosten_fahrzeuge_personal_euro":0,"nachschiebekosten_euro":0,"strecke_info":{"ausserhalb_db_netz":false,"ungeprueft_kv":fal

## 4.1 First Successful API Test

The first test request for a passenger night-train configuration was successfully processed by the Trassenfinder API (HTTP 200).

**Test configuration**
- Route: Augsburg Hbf (MA) → Hannover Hbf (HH)
- Composition: REF-BUD-6
- Coach weight at 80% occupancy: 323.3 t
- Coach length: 158.4 m
- Maximum speed: 200 km/h
- Locomotive: BR 193 / Siemens Vectron (6193)
- Traffic type: `spfv_lok`

**Returned results**
- Energy consumption: 2,301 kWh
- Trassenfinder route distance: 567.7 km

This confirms that the adapted payload for passenger rail transport with locomotive is accepted by the API and returns an energy consumption value.

The realistic travel time has not yet been extracted because it is not available under the expected field in the API response. This will be checked separately.

In [10]:
# ============================================================
# TEST 1 – Extract relevant API results
# ============================================================

data = response.json()["result"]
route_result = data["gewichtete_route"]

summary = route_result["zusammenfassung"]

print("✓ Route successfully calculated")
print()
print("Energy consumption:", summary["energieverbrauch_kwh"], "kWh")
print("Route distance:", summary["weglaenge_hm"] / 10, "km")

print()
print("Available fields in 'zusammenfassung':")
print(list(summary.keys()))

✓ Route successfully calculated

Energy consumption: 2301 kWh
Route distance: 567.7 km

Available fields in 'zusammenfassung':
['fahrzeit_technisch_min', 'weglaenge_hm', 'energieverbrauch_kwh', 'marktsegment', 'trassenpreis_euro', 'stationspreis_euro', 'preis_energie_euro', 'kosten_fahrzeuge_personal_euro', 'nachschiebekosten_euro']


## 4.2 Inspecting Travel Time Information

The Trassenfinder API does not return `fahrzeit_realistisch_min` in the route summary for this passenger rail request.

The summary contains `fahrzeit_technisch_min`, but the realistic travel time may be stored elsewhere in the detailed route response.

We therefore inspect the available top-level route fields before defining the final extraction logic.

In [11]:
# ============================================================
# TEST 1 – Inspect route structure for travel time
# ============================================================

print("Available fields in 'gewichtete_route':")
print(list(route_result.keys()))

Available fields in 'gewichtete_route':
['routen_typ', 'routenpunkte', 'technische_abfahrt', 'zusammenfassung', 'maximalwerte', 'punkt_zu_punkt_geschwindigkeit_unzulaessig', 'boundingbox']


## 4.3 Inspecting Detailed Travel Time Information

The route summary provides the technical travel time, but not the realistic travel time.

The detailed route points contain `technische_fahrzeit_info`. We inspect the first and last route point to determine how travel time is represented in the detailed response.

In [12]:
# ============================================================
# TEST 1 – Inspect detailed travel time information
# ============================================================

route_points = route_result["routenpunkte"]

print("Number of route points:", len(route_points))
print()
print("First route point:")
print(route_points[0]["technische_fahrzeit_info"])
print()
print("Last route point:")
print(route_points[-1]["technische_fahrzeit_info"])

Number of route points: 118

First route point:
{'ankunft_min': 0, 'abfahrt_min': 0}

Last route point:
{'ankunft_min': 345, 'abfahrt_min': 345}


## 4.4 Travel Time Extraction

The detailed route points provide the travel time in minutes.

For the test route, the first route point has a departure time of 0 minutes and the final route point has an arrival time of 345 minutes.

Therefore, the technical travel time of the calculated route is **345 minutes**.

For the training dataset, the technical travel time will be stored as `travel_time_min`. The API does not provide a separate realistic travel time for this passenger rail configuration.

In [13]:
# ============================================================
# TEST 1 – Extract travel time
# ============================================================

travel_time_min = route_points[-1]["technische_fahrzeit_info"]["ankunft_min"]

print("✓ Travel time extracted")
print("Technical travel time:", travel_time_min, "min")

✓ Travel time extracted
Technical travel time: 345 min


## 5. API Query Function

The successful test request is now converted into a reusable function.

The function takes:
- a start station (DS100)
- an end station (DS100)
- one train composition

It sends the request to Trassenfinder and extracts the three variables needed for the energy model:

- energy consumption (kWh)
- route distance (km)
- technical travel time (min)

The function also checks the HTTP response before processing the result.

In [14]:
# ============================================================
# 5. API QUERY FUNCTION
# ============================================================

def query_trassenfinder(start_ds100, end_ds100, composition):
    """
    Query Trassenfinder for one route and one train composition.

    Returns:
        Dictionary with energy consumption, distance and travel time.
    """

    # Create a fresh payload from the existing template
    request_payload = payload.copy()

    # Set route
    request_payload["wegpunkte"][0]["betriebsstelle"]["ds100"] = str(start_ds100)
    request_payload["wegpunkte"][1]["betriebsstelle"]["ds100"] = str(end_ds100)

    # Set train composition
    request_payload["wegpunkte"][0]["zugcharakteristik"]["triebfahrzeug"]["hauptnummer"] = str(
        composition["trassenfinder_triebfahrzeug_hauptnummer"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzuglaenge_m"] = float(
        composition["coaches_length_m_wagenzuglaenge"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzugmasse_t"] = float(
        composition["coaches_gross_weight_80pct_t_wagenzugmasse"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenanzahl"] = int(
        composition["n_coaches"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["v_max"] = float(
        composition["v_max_kmh"]
    )

    # Send request
    response = requests.post(
        url,
        json=request_payload,
        headers=headers
    )

    # Stop if API rejects the request
    response.raise_for_status()

    # Extract route
    route_result = response.json()["result"]["gewichtete_route"]
    summary = route_result["zusammenfassung"]
    route_points = route_result["routenpunkte"]

    # Extract relevant variables
    energy_kwh = summary["energieverbrauch_kwh"]
    distance_km = summary["weglaenge_hm"] / 10
    travel_time_min = route_points[-1]["technische_fahrzeit_info"]["ankunft_min"]

    return {
        "energy_kwh": energy_kwh,
        "distance_km": distance_km,
        "travel_time_min": travel_time_min
    }


print("✓ Query function created")

✓ Query function created


In [15]:
# Test the reusable query function

test_route = routes.iloc[0]
test_composition = compositions.iloc[0]

test_result = query_trassenfinder(
    test_route["start_ds100"],
    test_route["end_ds100"],
    test_composition
)

print("✓ Test successful")
print(test_result)

✓ Test successful
{'energy_kwh': 2301, 'distance_km': 567.7, 'travel_time_min': 345}


## 6. Collect Training Data

Each of the 96 active night-train route segments is queried with all 8 standard train compositions.

This results in up to **768 API requests** (96 routes × 8 compositions).

A failed request is logged and skipped so that one problematic route does not stop the entire data collection.

In [16]:
# ============================================================
# 6. COLLECT TRAINING DATA
# ============================================================

results = []
failed_requests = []

total_requests = len(routes) * len(compositions)

print(f"Starting data collection: {total_requests} requests")
print()

for route_index, route in routes.iterrows():

    for composition_index, composition in compositions.iterrows():

        try:
            result = query_trassenfinder(
                route["start_ds100"],
                route["end_ds100"],
                composition
            )

            results.append({
                "route_name": route["route_name"],
                "start_stop_name": route["start_stop_name"],
                "start_ds100": route["start_ds100"],
                "end_stop_name": route["end_stop_name"],
                "end_ds100": route["end_ds100"],
                "composition_id": composition["composition_id"],
                "n_coaches": composition["n_coaches"],
                "weight_t": composition["coaches_gross_weight_80pct_t_wagenzugmasse"],
                "length_m": composition["coaches_length_m_wagenzuglaenge"],
                "v_max_kmh": composition["v_max_kmh"],
                "energy_kwh": result["energy_kwh"],
                "distance_km": result["distance_km"],
                "travel_time_min": result["travel_time_min"]
            })

        except Exception as e:

            failed_requests.append({
                "route_name": route["route_name"],
                "start_ds100": route["start_ds100"],
                "end_ds100": route["end_ds100"],
                "composition_id": composition["composition_id"],
                "error": str(e)
            })

    print(
        f"Route {route_index + 1}/{len(routes)} finished | "
        f"Successful: {len(results)} | "
        f"Failed: {len(failed_requests)}"
    )

print()
print("✓ Data collection finished")
print("Successful requests:", len(results))
print("Failed requests:", len(failed_requests))

Starting data collection: 768 requests

Route 1/96 finished | Successful: 8 | Failed: 0
Route 2/96 finished | Successful: 16 | Failed: 0
Route 3/96 finished | Successful: 16 | Failed: 8
Route 4/96 finished | Successful: 24 | Failed: 8
Route 5/96 finished | Successful: 32 | Failed: 8
Route 6/96 finished | Successful: 32 | Failed: 16
Route 7/96 finished | Successful: 32 | Failed: 24
Route 8/96 finished | Successful: 32 | Failed: 32
Route 9/96 finished | Successful: 40 | Failed: 32
Route 10/96 finished | Successful: 48 | Failed: 32
Route 11/96 finished | Successful: 48 | Failed: 40
Route 12/96 finished | Successful: 48 | Failed: 48
Route 13/96 finished | Successful: 56 | Failed: 48


KeyboardInterrupt: 

## 6.1 Inspect Failed Requests

A substantial number of requests failed during the initial collection test.

Before continuing with all 96 route segments, the failed requests are inspected to determine whether the errors are caused by specific routes, train compositions, or API restrictions.

In [17]:
# ============================================================
# 6.1 INSPECT FAILED REQUESTS
# ============================================================

print("Number of failed requests:", len(failed_requests))
print()

for failure in failed_requests[:10]:
    print(failure)

Number of failed requests: 48

{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-BUD-6', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-COUCH-6', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-BAL-9', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-COUCH-10', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-BUD-12', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'REF-PREM-12', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'NEW-BAL-7', 'error': "'result'"}
{'route_name': '1856 = 1857', 'start_ds100': 'AH', 'end_ds100': 'AA', 'composition_id': 'NEW-BAL-14', 'erro

## 6.2 Improve API Error Logging

Some requests do not return a successful route. The current function hides the actual API response.

The query function is therefore extended to record the HTTP status and API error message. This allows failed routes to be diagnosed without stopping the complete data collection.

In [18]:
# ============================================================
# 6.2 TEST API ERROR RESPONSE
# ============================================================

def test_failed_request(start_ds100, end_ds100, composition):

    request_payload = payload.copy()

    request_payload["wegpunkte"][0]["betriebsstelle"]["ds100"] = str(start_ds100)
    request_payload["wegpunkte"][1]["betriebsstelle"]["ds100"] = str(end_ds100)

    request_payload["wegpunkte"][0]["zugcharakteristik"]["triebfahrzeug"]["hauptnummer"] = str(
        composition["trassenfinder_triebfahrzeug_hauptnummer"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzuglaenge_m"] = float(
        composition["coaches_length_m_wagenzuglaenge"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzugmasse_t"] = float(
        composition["coaches_gross_weight_80pct_t_wagenzugmasse"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenanzahl"] = int(
        composition["n_coaches"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["v_max"] = float(
        composition["v_max_kmh"]
    )

    response = requests.post(
        url,
        json=request_payload,
        headers=headers
    )

    print("Status:", response.status_code)
    print()
    print("API response:")
    print(response.text[:3000])

In [19]:
# Test the first failed route

failed = failed_requests[0]

failed_composition = compositions[
    compositions["composition_id"] == failed["composition_id"]
].iloc[0]

test_failed_request(
    failed["start_ds100"],
    failed["end_ds100"],
    failed_composition
)

Status: 200

API response:
{"failure":{"id":"1d97969a6a45c22f","message":"Es konnte keine Route von Hamburg Hbf (AH) nach Hamburg-Altona (AA) gefunden werden.","wegpunkt_index":0,"wegpunkt_details":[],"details":[{"ds100":"AE  F","streckennummer":1232,"ursachen":{"gesperrt_durch_baumassnahme":true}},{"ds100":"AE  O","streckennummer":1220,"ursachen":{"gesperrt_durch_baumassnahme":true}},{"ds100":"AH","streckennummer":6100,"ursachen":{"max_streckenklasse":"D3","gesperrt_durch_baumassnahme":true}}],"boundingbox":[7521.407931178794,47073.597554505905,70488.01938166896,124576.98731016372]}}


## 6.3 Handle Unavailable Routes

Trassenfinder can return HTTP 200 even when no valid route can be calculated.

In this case, the response contains `failure` instead of `result`.

These cases are recorded as failed requests and skipped. The successful requests remain part of the training dataset.

In [20]:
# ============================================================
# 6.3 UPDATED API QUERY FUNCTION
# ============================================================

def query_trassenfinder(start_ds100, end_ds100, composition):
    """
    Query Trassenfinder for one route and one train composition.

    Returns:
        Dictionary with energy consumption, distance and travel time.

    Raises:
        ValueError: if Trassenfinder cannot calculate the route.
    """

    request_payload = payload.copy()

    # Set route
    request_payload["wegpunkte"][0]["betriebsstelle"]["ds100"] = str(start_ds100)
    request_payload["wegpunkte"][1]["betriebsstelle"]["ds100"] = str(end_ds100)

    # Set train composition
    request_payload["wegpunkte"][0]["zugcharakteristik"]["triebfahrzeug"]["hauptnummer"] = str(
        composition["trassenfinder_triebfahrzeug_hauptnummer"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzuglaenge_m"] = float(
        composition["coaches_length_m_wagenzuglaenge"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenzugmasse_t"] = float(
        composition["coaches_gross_weight_80pct_t_wagenzugmasse"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["wagenanzahl"] = int(
        composition["n_coaches"]
    )

    request_payload["wegpunkte"][0]["zugcharakteristik"]["v_max"] = float(
        composition["v_max_kmh"]
    )

    # Send request
    response = requests.post(
        url,
        json=request_payload,
        headers=headers
    )

    response_data = response.json()

    # Handle API-level route failure
    if "failure" in response_data:
        raise ValueError(response_data["failure"]["message"])

    # Handle HTTP errors
    response.raise_for_status()

    # Extract successful route
    route_result = response_data["result"]["gewichtete_route"]
    summary = route_result["zusammenfassung"]
    route_points = route_result["routenpunkte"]

    return {
        "energy_kwh": summary["energieverbrauch_kwh"],
        "distance_km": summary["weglaenge_hm"] / 10,
        "travel_time_min": route_points[-1]["technische_fahrzeit_info"]["ankunft_min"]
    }


print("✓ Updated query function created")

✓ Updated query function created


In [21]:
# Test the updated function with the previously failed route

failed = failed_requests[0]

failed_composition = compositions[
    compositions["composition_id"] == failed["composition_id"]
].iloc[0]

try:
    result = query_trassenfinder(
        failed["start_ds100"],
        failed["end_ds100"],
        failed_composition
    )

    print("✓ Route successful")
    print(result)

except Exception as e:
    print("✓ Route correctly identified as unavailable")
    print("Reason:", e)

✓ Route correctly identified as unavailable
Reason: Es konnte keine Route von Hamburg Hbf (AH) nach Hamburg-Altona (AA) gefunden werden.


## 7. Full Training Data Collection

The API query function now handles both successful routes and unavailable routes.

We now repeat the process for all 96 night-train route segments and all 8 train compositions.

Unavailable routes are logged and skipped instead of stopping the collection.

In [22]:
# ============================================================
# 7. FULL TRAINING DATA COLLECTION
# ============================================================

results = []
failed_requests = []

total_requests = len(routes) * len(compositions)

print(f"Starting data collection: {total_requests} requests")
print()

for route_index, route in routes.iterrows():

    for composition_index, composition in compositions.iterrows():

        try:
            result = query_trassenfinder(
                route["start_ds100"],
                route["end_ds100"],
                composition
            )

            results.append({
                "route_name": route["route_name"],
                "start_stop_name": route["start_stop_name"],
                "start_ds100": route["start_ds100"],
                "end_stop_name": route["end_stop_name"],
                "end_ds100": route["end_ds100"],
                "composition_id": composition["composition_id"],
                "n_coaches": composition["n_coaches"],
                "weight_t": composition["coaches_gross_weight_80pct_t_wagenzugmasse"],
                "length_m": composition["coaches_length_m_wagenzuglaenge"],
                "v_max_kmh": composition["v_max_kmh"],
                "energy_kwh": result["energy_kwh"],
                "distance_km": result["distance_km"],
                "travel_time_min": result["travel_time_min"]
            })

        except Exception as e:

            failed_requests.append({
                "route_name": route["route_name"],
                "start_ds100": route["start_ds100"],
                "end_ds100": route["end_ds100"],
                "composition_id": composition["composition_id"],
                "error": str(e)
            })

    print(
        f"Route {route_index + 1}/{len(routes)} finished | "
        f"Successful: {len(results)} | "
        f"Failed: {len(failed_requests)}"
    )

print()
print("✓ Data collection finished")
print("Successful requests:", len(results))
print("Failed requests:", len(failed_requests))

Starting data collection: 768 requests

Route 1/96 finished | Successful: 8 | Failed: 0
Route 2/96 finished | Successful: 16 | Failed: 0
Route 3/96 finished | Successful: 16 | Failed: 8
Route 4/96 finished | Successful: 24 | Failed: 8
Route 5/96 finished | Successful: 32 | Failed: 8
Route 6/96 finished | Successful: 32 | Failed: 16
Route 7/96 finished | Successful: 32 | Failed: 24
Route 8/96 finished | Successful: 32 | Failed: 32
Route 9/96 finished | Successful: 40 | Failed: 32
Route 10/96 finished | Successful: 48 | Failed: 32
Route 11/96 finished | Successful: 48 | Failed: 40
Route 12/96 finished | Successful: 48 | Failed: 48
Route 13/96 finished | Successful: 56 | Failed: 48
Route 14/96 finished | Successful: 64 | Failed: 48
Route 15/96 finished | Successful: 72 | Failed: 48
Route 16/96 finished | Successful: 80 | Failed: 48
Route 17/96 finished | Successful: 88 | Failed: 48
Route 18/96 finished | Successful: 96 | Failed: 48
Route 19/96 finished | Successful: 104 | Failed: 48
Route

## Trassenfinder Data Collection

The Trassenfinder API was queried for all 96 selected night-train route segments and 8 train compositions.

Total requests: 768

The API returned successful energy-consumption results for 448 requests.
320 requests failed because no valid route could be calculated for the respective route/composition combination.

Failed requests are retained separately for quality control and are not used as training samples.

## Save Trassenfinder Results

The successful Trassenfinder API requests are saved as the training dataset.
Failed requests are stored separately and excluded from the training data.

In [24]:
# Check which variables contain our collected results

print([name for name in globals() if "result" in name.lower()])
print([name for name in globals() if "fail" in name.lower()])

['route_result', 'test_result', 'results', 'result']
['failed_requests', 'failure', 'test_failed_request', 'failed', 'failed_composition']


## Save Trassenfinder Results

Successful API results are saved as the training dataset.
Failed API requests are saved separately for quality control.

In [25]:
# Save successful and failed results

successful_df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed_requests)

successful_df.to_csv(
    "../data/processed/trassenfinder_energy_results.csv",
    index=False
)

failed_df.to_csv(
    "../data/processed/trassenfinder_failed_requests.csv",
    index=False
)

print("✓ Successful results saved:", len(successful_df))
print("✓ Failed requests saved:", len(failed_df))

OSError: Cannot save file into a non-existent directory: '../data/processed'

In [26]:
import os

print("Aktueller Arbeitsordner:")
print(os.getcwd())

print("\nGesuchte Dateien:")

for filename in [
    "trassenfinder_energy_results.csv",
    "trassenfinder_failed_requests.csv"
]:
    path = os.path.abspath("../data/processed/" + filename)
    print("\n", filename)
    print("Pfad:", path)
    print("Existiert:", os.path.exists(path))

Aktueller Arbeitsordner:
/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/notebooks

Gesuchte Dateien:

 trassenfinder_energy_results.csv
Pfad: /Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/data/processed/trassenfinder_energy_results.csv
Existiert: False

 trassenfinder_failed_requests.csv
Pfad: /Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/data/processed/trassenfinder_failed_requests.csv
Existiert: False


In [27]:
import os

project_root = "/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network"

for root, dirs, files in os.walk(project_root):
    for file in files:
        if file in [
            "trassenfinder_energy_results.csv",
            "trassenfinder_failed_requests.csv"
        ]:
            print(os.path.join(root, file))

In [28]:
print("results vorhanden:", "results" in globals())
print("Anzahl results:", len(results) if "results" in globals() else "NICHT VORHANDEN")

print()
print("failed_requests vorhanden:", "failed_requests" in globals())
print("Anzahl failed_requests:", len(failed_requests) if "failed_requests" in globals() else "NICHT VORHANDEN")

results vorhanden: True
Anzahl results: 448

failed_requests vorhanden: True
Anzahl failed_requests: 320


In [29]:
import os
import pandas as pd

# Zielordner
output_dir = "/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/data/processed"

# Ordner sicherstellen
os.makedirs(output_dir, exist_ok=True)

# DataFrames aus den noch vorhandenen Ergebnissen erstellen
successful_df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed_requests)

# Absolute Dateipfade
successful_path = os.path.join(
    output_dir,
    "trassenfinder_energy_results.csv"
)

failed_path = os.path.join(
    output_dir,
    "trassenfinder_failed_requests.csv"
)

# Speichern
successful_df.to_csv(successful_path, index=False)
failed_df.to_csv(failed_path, index=False)

# DIREKT überprüfen
print("Erfolgreiche Daten:")
print("Datei:", successful_path)
print("Existiert:", os.path.exists(successful_path))
print("Anzahl Zeilen:", len(successful_df))

print()

print("Fehlgeschlagene Anfragen:")
print("Datei:", failed_path)
print("Existiert:", os.path.exists(failed_path))
print("Anzahl Zeilen:", len(failed_df))

Erfolgreiche Daten:
Datei: /Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/data/processed/trassenfinder_energy_results.csv
Existiert: True
Anzahl Zeilen: 448

Fehlgeschlagene Anfragen:
Datei: /Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/backend/models/energy/data/processed/trassenfinder_failed_requests.csv
Existiert: True
Anzahl Zeilen: 320


In [30]:
# ============================================================
# 8.1 – FIRST DATA QUALITY CHECK
# ============================================================

print("Successful training samples:", len(successful_df))
print("Failed requests:", len(failed_df))

print("\nColumns:")
print(successful_df.columns.tolist())

print("\nFirst 5 rows:")
display(successful_df.head())

print("\nMissing values:")
print(successful_df.isnull().sum())

print("\nNumeric summary:")
display(successful_df.describe())

Successful training samples: 448
Failed requests: 320

Columns:
['route_name', 'start_stop_name', 'start_ds100', 'end_stop_name', 'end_ds100', 'composition_id', 'n_coaches', 'weight_t', 'length_m', 'v_max_kmh', 'energy_kwh', 'distance_km', 'travel_time_min']

First 5 rows:


,route_name,start_stop_name,start_ds100,end_stop_name,end_ds100,composition_id,n_coaches,weight_t,length_m,v_max_kmh,energy_kwh,distance_km,travel_time_min
0,1856 = 1857,Augsburg Hbf,MA,Hannover Hbf,HH,REF-BUD-6,6,323.3,158.4,200,2301,567.7,345
1,1856 = 1857,Augsburg Hbf,MA,Hannover Hbf,HH,REF-COUCH-6,6,313.0,158.4,200,2274,567.7,345
2,1856 = 1857,Augsburg Hbf,MA,Hannover Hbf,HH,REF-BAL-9,9,501.4,237.6,200,2979,567.6,346
3,1856 = 1857,Augsburg Hbf,MA,Hannover Hbf,HH,REF-COUCH-10,10,532.2,264.0,200,3098,572.8,353
4,1856 = 1857,Augsburg Hbf,MA,Hannover Hbf,HH,REF-BUD-12,12,636.0,316.8,200,3485,572.9,354



Missing values:
route_name         0
start_stop_name    0
start_ds100        0
end_stop_name      0
end_ds100          0
composition_id     0
n_coaches          0
weight_t           0
length_m           0
v_max_kmh          0
energy_kwh         0
distance_km        0
travel_time_min    0
dtype: int64

Numeric summary:


,n_coaches,weight_t,length_m,v_max_kmh,energy_kwh,distance_km,travel_time_min
count,448.000000,448.000000,448.000000,448.000000,448.000000,448.000000,448.000000
mean,9.500000,484.862500,251.137500,207.500000,984.868304,173.889732,107.486607
std,2.831589,138.069145,75.014953,13.004904,1063.849461,186.168898,113.919080
min,6.000000,313.000000,158.400000,200.000000,30.000000,3.800000,4.000000
25%,6.750000,320.925000,178.875000,200.000000,266.750000,49.900000,31.750000
50%,9.500000,516.800000,250.800000,200.000000,509.500000,97.650000,62.000000
75%,12.000000,628.600000,316.800000,207.500000,1249.500000,212.775000,132.750000
max,14.000000,636.000000,371.400000,230.000000,5257.000000,817.200000,498.000000


In [31]:
# ============================================================
# 8.2 – DATA STRUCTURE & COVERAGE CHECK
# ============================================================

print("Number of unique routes:", successful_df["route_name"].nunique())
print("Number of unique compositions:", successful_df["composition_id"].nunique())

print("\nSuccessful samples per route:")
display(
    successful_df.groupby("route_name")
    .size()
    .value_counts()
    .sort_index()
)

print("\nSuccessful samples per composition:")
display(
    successful_df.groupby("composition_id")
    .size()
    .sort_values()
)

print("\nDistance range per route:")
distance_check = (
    successful_df
    .groupby(["route_name", "start_stop_name", "end_stop_name"])
    ["distance_km"]
    .agg(["min", "max", "count"])
    .reset_index()
)

display(distance_check.sort_values("max", ascending=False).head(20))

Number of unique routes: 21
Number of unique compositions: 8

Successful samples per route:


8     8
16    3
24    5
32    2
40    1
56    2
Name: count, dtype: int64


Successful samples per composition:


composition_id
NEW-BAL-14      56
NEW-BAL-7       56
REF-BAL-9       56
REF-BUD-12      56
REF-BUD-6       56
REF-COUCH-10    56
REF-COUCH-6     56
REF-PREM-12     56
dtype: int64


Distance range per route:


,route_name,start_stop_name,end_stop_name,min,max,count
8,304 = 305,München Ost,Hamburg Hbf,817.2,817.2,8
7,302/322 = 303/323,München Ost,Hannover Hbf,635.5,638.9,8
0,1856 = 1857,Augsburg Hbf,Hannover Hbf,567.6,572.9,8
23,Canopus,Leipzig Hbf,Karlsruhe Hbf,502.9,502.9,8
5,1858 = 1859,Karlsruhe Hbf,Hannover Hbf,497.9,501.3,8
47,NJ 408 = NJ 409,Leipzig Hbf,Mannheim Hbf,452.3,452.3,8
14,332/1372 = 333/1373,Mainz Hbf,München Ost,442.8,442.8,8
44,NJ 40421 = NJ 40490,Würzburg Hbf,Oberhausen Hbf,426.3,426.4,8
15,332/1372 = 333/1373,München Ost,Darmstadt Hbf,414.2,415.0,8
12,320/324 = 321/325,München Ost,Fulda,394.1,394.1,8


## 8.3 Analysis of Failed Route Requests

Before using the successful Trassenfinder results as training data, the failed requests are analysed separately.

The Trassenfinder API may return a failure for several reasons, including:

- temporary infrastructure restrictions or construction works
- route-specific operational constraints
- incompatibility between the train composition and the requested route
- invalid or unsuitable station/route combinations

The purpose of this analysis is to determine whether the 320 failed requests represent systematic limitations of the input data or temporary/route-specific restrictions.

Failed requests are not included in the training dataset.

In [32]:
# ============================================================
# 8.3 – ANALYSIS OF FAILED ROUTE REQUESTS
# ============================================================

failed_route_summary = (
    failed_df
    .groupby(["route_name", "start_ds100", "end_ds100"])
    .agg(
        failed_combinations=("composition_id", "count"),
        errors=("error", lambda x: " | ".join(x.unique()))
    )
    .reset_index()
)

print("Number of unique failed routes:")
print(len(failed_route_summary))

print("\nFailed routes:")
display(failed_route_summary)

Number of unique failed routes:
39

Failed routes:


,route_name,start_ds100,end_ds100,failed_combinations,errors
0,1856 = 1857,AH,AA,8,Es konnte keine Route von Hamburg Hbf (AH) nac...
1,1856 = 1857,MP,MS,8,400 Client Error: Bad Request for url: https:/...
2,1856 = 1857,MS,MOP,8,400 Client Error: Bad Request for url: https:/...
3,1858 = 1859,RF,RK,8,400 Client Error: Bad Request for url: https:/...
4,1858 = 1859,RLR,RF,8,400 Client Error: Bad Request for url: https:/...
5,332/1372 = 333/1373,FD,KKO,8,400 Client Error: Bad Request for url: https:/...
6,332/1372 = 333/1373,KKO,FMZ,8,400 Client Error: Bad Request for url: https:/...
7,332/1372 = 333/1373,KKO,KK,8,400 Client Error: Bad Request for url: https:/...
8,334/1374 = 335/1375,KKO,NAH,8,400 Client Error: Bad Request for url: https:/...
9,452 = 453,DN,BLS,8,400 Client Error: Bad Request for url: https:/...


## 8.4 Inspection of HTTP 400 Errors

A large proportion of the failed requests returned HTTP 400 (`Bad Request`).

The current query function records the HTTP error but does not preserve the detailed response returned by the Trassenfinder API. Therefore, the exact reason for these failures is not yet known.

Before excluding the affected route segments from the training dataset, representative HTTP 400 responses are inspected directly.

This is necessary to distinguish between:

- invalid request parameters,
- unsupported station or route combinations,
- train-specific restrictions,
- temporary infrastructure restrictions, and
- other API-side validation errors.

The failed routes will only be excluded from the training dataset after their failure reasons have been assessed.

In [33]:
# ============================================================
# 8.4 – INSPECT HTTP 400 ERRORS
# ============================================================

# Select the first failed request that returned an HTTP 400 error
http_400_failure = next(
    failure
    for failure in failed_requests
    if "400 Client Error" in failure["error"]
)

print("Route:")
print(
    http_400_failure["start_ds100"],
    "→",
    http_400_failure["end_ds100"]
)

print("\nComposition:")
print(http_400_failure["composition_id"])

Route:
MS → MOP

Composition:
REF-BUD-6


In [34]:
test_failed_request(
    "MS",
    "MOP",
    compositions[
        compositions["composition_id"] == "REF-BUD-6"
    ].iloc[0]
)

Status: 400

API response:
{"message":"Ungültige Anfrage","details":["Ungültige Betriebsstelle 'MS' (Mutter: true)"],"error_type":"invalid_request"}


In [36]:
# ============================================================
# 8.5 IDENTIFY DS100 CODES ASSOCIATED WITH HTTP 400 ERRORS
# ============================================================

from collections import Counter

invalid_points = Counter()

for failure in failed_requests:

    if "400 Client Error" in failure["error"]:
        invalid_points[failure["start_ds100"]] += 1
        invalid_points[failure["end_ds100"]] += 1

print("DS100 codes occurring in HTTP 400 failures:")
print()

for ds100, count in invalid_points.most_common():
    print(f"{ds100}: {count}")

DS100 codes occurring in HTTP 400 failures:

KKO: 56
FKW: 32
RF: 24
KK: 24
BLS: 24
KA: 24
FH  N: 24
MS: 16
MOP: 16
RK: 16
RLR: 16
KKSU: 16
RBB: 16
RO: 16
UE  P: 16
FFU: 16
FFS: 16
BPAF: 16
TGO: 16
HG: 16
LW: 16
LBT: 16
EHM: 16
KBB: 16
MP: 8
nan: 8
FD: 8
FMZ: 8
NAH: 8
DN: 8
KD: 8
AH: 8
LL: 8
FFLF: 8
TS: 8
TU: 8
NWH: 8
EE: 8
LH: 8
EMST: 8
KB: 8
KKER: 8


## 8.6 Validating the DS100 Identifier Issue

The HTTP 400 errors occur across a large number of different DS100 identifiers. This suggests that the problem is not limited to a single operating point.

Instead of manually correcting all affected identifiers, a controlled comparison is performed between:

1. a route that was successfully calculated by Trassenfinder, and
2. a route segment that returned HTTP 400.

This test determines whether the issue is caused by the operating-point identifiers used in the route dataset or by another systematic payload parameter.

In [37]:
# ============================================================
# 8.6 CONTROLLED COMPARISON
# ============================================================

# Known successful route
successful_route = routes.iloc[0]
successful_composition = compositions.iloc[0]

print("KNOWN SUCCESSFUL ROUTE")
print("----------------------")
print("Start:", successful_route["start_ds100"])
print("End:", successful_route["end_ds100"])
print("Composition:", successful_composition["composition_id"])

try:
    success_test = query_trassenfinder(
        successful_route["start_ds100"],
        successful_route["end_ds100"],
        successful_composition
    )

    print("Result: SUCCESS")
    print(success_test)

except Exception as e:
    print("Result: FAILED")
    print(e)


print()
print("=" * 60)
print()


# Known HTTP 400 route
failed_test = failed_requests[1]

failed_composition = compositions[
    compositions["composition_id"] == failed_test["composition_id"]
].iloc[0]

print("KNOWN FAILED ROUTE")
print("------------------")
print("Start:", failed_test["start_ds100"])
print("End:", failed_test["end_ds100"])
print("Composition:", failed_test["composition_id"])

try:
    failed_test_result = query_trassenfinder(
        failed_test["start_ds100"],
        failed_test["end_ds100"],
        failed_composition
    )

    print("Result: SUCCESS")
    print(failed_test_result)

except Exception as e:
    print("Result: FAILED")
    print(e)

KNOWN SUCCESSFUL ROUTE
----------------------
Start: MA
End: HH
Composition: REF-BUD-6
Result: SUCCESS
{'energy_kwh': 2301, 'distance_km': 567.7, 'travel_time_min': 345}


KNOWN FAILED ROUTE
------------------
Start: AH
End: AA
Composition: REF-COUCH-6
Result: FAILED
Es konnte keine Route von Hamburg Hbf (AH) nach Hamburg-Altona (AA) gefunden werden.


## 8.7 Inspecting a Genuine HTTP 400 Error

The previous comparison included a route that returned an API-level routing failure with HTTP 200. This represents a genuine unavailable route and is therefore different from the HTTP 400 errors.

A representative HTTP 400 case is therefore inspected separately.

The route `MS → MOP` is used because the API explicitly identifies `MS` as an invalid operating point (`Betriebsstelle`). This allows us to distinguish invalid operating-point identifiers from routes that are simply unavailable due to infrastructure conditions.

In [38]:
# ============================================================
# 8.7 INSPECT A GENUINE HTTP 400 ERROR
# ============================================================

start_ds100 = "MS"
end_ds100 = "MOP"

composition_test = compositions.iloc[0]

print("Route:")
print(f"{start_ds100} → {end_ds100}")
print("Composition:", composition_test["composition_id"])
print()

test_failed_request(
    start_ds100,
    end_ds100,
    composition_test
)

Route:
MS → MOP
Composition: REF-BUD-6

Status: 400

API response:
{"message":"Ungültige Anfrage","details":["Ungültige Betriebsstelle 'MS' (Mutter: true)"],"error_type":"invalid_request"}


## 8.8 Summary of Failed Requests

Of 768 API requests, 448 were successful and 320 failed.

- 8 requests: no valid route available.
- 312 requests: invalid operating-point identifiers (`Betriebsstellen`).

The 448 successful requests form the current training dataset.

## 9. Data Quality Check

Before developing the regression model, the successful Trassenfinder results are checked for completeness, duplicates and basic plausibility.

In [39]:
# ============================================================
# 9.1 DATA QUALITY CHECK
# ============================================================

training_df = pd.read_csv(
    "../data/processed/trassenfinder_energy_results.csv"
)

print("Training samples:", len(training_df))
print("Unique routes:", training_df["route_name"].nunique())
print("Unique compositions:", training_df["composition_id"].nunique())
print()

# Missing values
print("Missing values:")
print(training_df.isna().sum())
print()

# Duplicate rows
print("Duplicate rows:", training_df.duplicated().sum())
print()

# Check expected compositions
print("Samples per composition:")
print(training_df["composition_id"].value_counts().sort_index())
print()

# Basic plausibility checks
print("Invalid / non-positive values:")
print("Energy <= 0:", (training_df["energy_kwh"] <= 0).sum())
print("Distance <= 0:", (training_df["distance_km"] <= 0).sum())
print("Travel time <= 0:", (training_df["travel_time_min"] <= 0).sum())

Training samples: 448
Unique routes: 21
Unique compositions: 8

Missing values:
route_name         0
start_stop_name    0
start_ds100        0
end_stop_name      0
end_ds100          0
composition_id     0
n_coaches          0
weight_t           0
length_m           0
v_max_kmh          0
energy_kwh         0
distance_km        0
travel_time_min    0
dtype: int64

Duplicate rows: 0

Samples per composition:
composition_id
NEW-BAL-14      56
NEW-BAL-7       56
REF-BAL-9       56
REF-BUD-12      56
REF-BUD-6       56
REF-COUCH-10    56
REF-COUCH-6     56
REF-PREM-12     56
Name: count, dtype: int64

Invalid / non-positive values:
Energy <= 0: 0
Distance <= 0: 0
Travel time <= 0: 0


## 9.2 Final Dataset Summary

The Trassenfinder query successfully generated a clean training dataset for further energy model development.

Dataset characteristics:

- 448 successful API samples
- 21 unique railway routes
- 8 train compositions
- No missing values
- No duplicate entries
- Valid energy, distance and travel time values for all samples

The dataset contains the input variables required for future regression-based energy modelling.
The model development step is performed separately.

In [40]:
# ============================================================
# 9.2 FINAL DATASET SUMMARY
# ============================================================

print("Final training dataset")
print("---------------------")
print("Samples:", len(training_df))
print("Routes:", training_df["route_name"].nunique())
print("Compositions:", training_df["composition_id"].nunique())

print()
print("Variables:")
for column in training_df.columns:
    print("-", column)

Final training dataset
---------------------
Samples: 448
Routes: 21
Compositions: 8

Variables:
- route_name
- start_stop_name
- start_ds100
- end_stop_name
- end_ds100
- composition_id
- n_coaches
- weight_t
- length_m
- v_max_kmh
- energy_kwh
- distance_km
- travel_time_min


## 10. Handover Documentation

The Trassenfinder API workflow has been completed and provides a cleaned training dataset for further energy model development.

### Generated files

Successful API results:

`data/processed/trassenfinder_energy_results.csv`

Contains:
- route information
- DS100 start and end identifiers
- train composition parameters
- energy consumption
- route distance
- technical travel time

Failed API requests:

`data/processed/trassenfinder_failed_requests.csv`

Contains:
- unavailable routes
- invalid operating-point requests
- error messages for quality control

### Dataset status

The final training dataset contains:

- 448 successful samples
- 21 unique routes
- 8 train compositions
- complete values without missing data

The dataset is ready for regression-based energy model development.

## 10.1 Reproducibility

The dataset can be regenerated by executing the Trassenfinder API query notebook:

`notebooks/Trassenfinder_API_query.ipynb`

The notebook:
1. loads route and train composition data,
2. sends requests to the Trassenfinder API,
3. extracts energy consumption, distance and travel time,
4. saves the resulting training dataset.